In [1]:
import os
import pandas as pd

def build_path_format_dir(df, image_root, output_dir):
    """Construit un dossier au format 'path' pour ketos compile.

    Pour chaque ligne du dataframe, cree un lien symbolique vers l'image
    source et un fichier .gt.txt contenant la transcription, sous le meme
    nom de base (requis par `ketos compile -f path`).

    Args:
        df: DataFrame avec colonnes 'image_path' (relatif a image_root)
            et 'text'.
        image_root: Repertoire racine contenant les images (dataset_nlp/).
        output_dir: Repertoire de sortie a creer.

    Returns:
        Nombre de paires (image, texte) effectivement creees.
    """
    os.makedirs(output_dir, exist_ok=True)
    n = 0
    for _, row in df.iterrows():
        src_img = os.path.join(image_root, row['image_path'])
        if not os.path.exists(src_img):
            continue
        basename = os.path.splitext(os.path.basename(row['image_path']))[0]
        dst_img = os.path.join(output_dir, f"{basename}.png")
        dst_txt = os.path.join(output_dir, f"{basename}.gt.txt")

        if not os.path.exists(dst_img):
            os.symlink(os.path.abspath(src_img), dst_img)
        with open(dst_txt, 'w', encoding='utf-8') as f:
            f.write(str(row['text']))
        n += 1
    return n


train_df = pd.read_csv("../dataset_nlp/splits/train.csv")
val_df = pd.read_csv("../dataset_nlp/splits/val.csv")

n_train = build_path_format_dir(train_df, "../dataset_nlp", "../dataset_nlp/ketos_train")
n_val = build_path_format_dir(val_df, "../dataset_nlp", "../dataset_nlp/ketos_val")

print(f"Train : {n_train}/{len(train_df)} paires creees")
print(f"Val   : {n_val}/{len(val_df)} paires creees")

Train : 23444/23444 paires creees
Val   : 5954/5954 paires creees


In [2]:
%%bash
mkdir -p ../dataset_nlp/binary_data
cd ../dataset_nlp/ketos_train && ketos compile -f path -o ../binary_data/train.arrow *.png

Extracting lines ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 23444/23444 0:00:00 0:18:381 0:18:380:18:290m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0/0 -:--:-- -:--:--
Output file written to ../binary_data/train.arrow


In [3]:
%%bash
ls ../models/
ketos train --help | head -60

Tridis_Medieval_EarlyModern.mlmodel
cremma-medieval.mlmodel
finetune_cremma_test
mon_modele_finetune
Usage: ketos train [OPTIONS] [GROUND_TRUTH]...

  Trains a model from image-text pairs.

Options:
  -B, --batch-size INTEGER        batch sample size  [default: 1]
  --pad INTEGER                   Left and right padding around lines
                                  [default: 16]
  -o, --output TEXT               Directory to save checkpoints into.
                                  [default: model]
  --weights-format TEXT           Output weights format.  [default:
                                  safetensors]
  -s, --spec TEXT                 VGSL spec of the network to train. CTC layer
                                  will be added automatically.  [default:
                                  [1,120,0,1 Cr3,13,32 Do0.1,2 Mp2,2 Cr3,13,32
                                  Do0.1,2 Mp2,2 Cr3,9,64 Do0.1,2 Mp2,2
                                  Cr3,9,64 Do0.1,2 S1(1x0)1,3 Lbx200 Do0.1,2
 

In [4]:
%%bash
ketos train --help | tail -60

                                  option.  [default: constant]
  -g, --gamma FLOAT               Decay factor for exponential, step, and
                                  reduceonplateau learning rate schedules
                                  [default: 0.1]
  -ss, --step-size INTEGER        Number of validation runs between learning
                                  rate decay for exponential and step LR
                                  schedules  [default: 10]
  --sched-patience INTEGER        Minimal number of validation runs between LR
                                  reduction for reduceonplateau LR schedule.
                                  [default: 5]
  --cos-max INTEGER               Epoch of minimal learning rate for cosine LR
                                  scheduler.  [default: 10]
  --cos-min-lr FLOAT              Minimal final learning rate for cosine LR
                                  scheduler.  [default: 1e-06]
  -p, --partition FLOAT           Ground truth dat

In [5]:
import pyarrow as pa
import json

def set_split(input_path, output_path, counts, train_val, validation_val, test_val):
    with pa.memory_map(input_path, 'r') as source:
        reader = pa.ipc.open_file(source)
        table = reader.read_all()
        raw_metadata = dict(reader.schema.metadata)

    lines_meta = json.loads(raw_metadata[b'lines'])
    lines_meta['counts'] = counts
    raw_metadata[b'lines'] = json.dumps(lines_meta).encode('utf-8')

    n = table.num_rows
    table = table.set_column(table.schema.get_field_index('train'), 'train', pa.array([train_val] * n, type=pa.bool_()))
    table = table.set_column(table.schema.get_field_index('validation'), 'validation', pa.array([validation_val] * n, type=pa.bool_()))
    table = table.set_column(table.schema.get_field_index('test'), 'test', pa.array([test_val] * n, type=pa.bool_()))
    table = table.replace_schema_metadata(raw_metadata)

    with pa.OSFile(output_path, 'wb') as sink:
        with pa.ipc.new_file(sink, table.schema) as writer:
            writer.write_table(table)

# train.arrow -> 100% train
set_split("../dataset_nlp/binary_data/train.arrow", "../dataset_nlp/binary_data/train_fixed.arrow",
          counts={"all": 23444, "train": 23444, "validation": 0, "test": 0},
          train_val=True, validation_val=False, test_val=False)

# val.arrow -> 100% validation
set_split("../dataset_nlp/binary_data/val.arrow", "../dataset_nlp/binary_data/val_fixed.arrow",
          counts={"all": 5954, "train": 0, "validation": 5954, "test": 0},
          train_val=False, validation_val=True, test_val=False)

print("✅ counts + colonnes corriges")

✅ counts + colonnes corriges


In [6]:
import pyarrow as pa, json

with pa.memory_map("../dataset_nlp/binary_data/train_fixed.arrow", 'r') as f:
    reader = pa.ipc.open_file(f)
    meta = json.loads(reader.schema.metadata[b'lines'])
    table = reader.read_all()

print("type:", meta['type'])
print("counts:", meta['counts'])
print("train col True count:", table.column('train').to_pylist().count(True))
print("val col True count:", table.column('validation').to_pylist().count(True))
print("sample ligne 0:", json.loads(reader.schema.metadata[b'lines'])['alphabet'])

type: kraken_recognition_bbox
counts: {'all': 23444, 'train': 23444, 'validation': 0, 'test': 0}
train col True count: 23444
val col True count: 0
sample ligne 0: {'L': 1148, 'o': 51957, 'y': 5460, 's': 73445, ',': 8279, ' ': 182259, 'p': 22232, 'a': 60843, 'r': 56793, 'l': 49616, 'g': 9010, 'c': 28784, 'e': 148656, 'd': 30187, 'D': 1248, 'i': 70158, 'u': 64544, 'F': 230, 'n': 61783, '.': 8725, 'S': 1051, 'v': 4467, 'f': 10796, 'à': 1482, 't': 70331, '’': 2782, 'm': 21376, 'b': 7302, 'é': 2313, 'J': 363, 'h': 7208, 'B': 493, 'è': 1010, 'M': 891, 'z': 5653, 'q': 11572, 'x': 2914, '-': 895, 'C': 1262, 'A': 891, 'j': 982, ';': 427, 'E': 2190, 'P': 1149, 'U': 181, 'H': 105, 'T': 470, 'V': 163, ':': 283, 'R': 596, '—': 73, 'G': 414, 'ï': 116, 'ù': 111, '«': 40, '!': 17, '»': 40, '(': 95, ')': 85, 'ç': 119, '[': 55, ']': 55, 'O': 327, 'N': 490, 'â': 3, 'Q': 460, 'ô': 1, '\xa0': 224, 'Y': 15, 'K': 5, '?': 7, 'X': 17, 'I': 736, '…': 12, 'ë': 2, "'": 140, 'œ': 3, '¬': 2, '̃': 5911, '\uf1ac': 45

In [7]:
import sys
sys.path.insert(0, '/root/htr-medieval-manuscripts-XIVe/venv/lib/python3.12/site-packages')

from kraken.lib.dataset import ArrowIPCRecognitionDataset

ds = ArrowIPCRecognitionDataset(split_filter='train')
ds.add('../dataset_nlp/binary_data/train_fixed.arrow')
print(f"Nombre de lignes dans le dataset : {len(ds)}")
print(f"seg_type : {ds.seg_type}")

Nombre de lignes dans le dataset : 23444
seg_type : bbox


In [8]:
from kraken.configs import VGSLRecognitionTrainingDataConfig
import inspect
print(inspect.getsource(VGSLRecognitionTrainingDataConfig))

class VGSLRecognitionTrainingDataConfig(RecognitionTrainingDataConfig):
    """
    Training data configuration for training VGSL recognition models.

    Arg:
        normalization (str, defaults to None):
            Unicode normalization
        normalize_whitespace (bool, defaults to True):
            Flag to normalize all whitespace in training data to U+0020.
        bidi_reordering (bool, defaults to True):
            Reorder code points according to the Unicode bidirectional
            algorithm. Set to L|R to override default text direction.
        legacy_polygons (bool, defaults to False):
            Whether to use the slow legacy polygon extractor for training.
        paddding (int, defaults to 16):
            Padding around start/end of line image.
    """
    def __init__(self, **kwargs):
        self.normalization = kwargs.pop('normalization', None)
        self.normalize_whitespace = kwargs.pop('normalize_whitespace', True)
        self.bidi_reordering = kwargs.po

In [9]:
from kraken.configs import RecognitionTrainingDataConfig
import inspect
print(inspect.getsource(RecognitionTrainingDataConfig))

class RecognitionTrainingDataConfig(TrainingDataConfig):
    """
    Configuration for recognition training data.

    Arg:
        > Text recognition parameters

        binary_dataset_split (bool, defaults to False):
            Flag to retrieve fixed splits from binary datasets.
        format_type (Literal['alto', 'page', 'xml', 'binary'], defaults to 'xml'):
            Format of the training data.
        codec: (Union[dict[str, Sequence[int]], Sequence[str], str], defaults to None):
            Codec mapping one or more Unicode code points to one or more
            integers.
    """
    def __init__(self, **kwargs):
        self.binary_dataset_split = kwargs.pop('binary_dataset_split', False)
        self.format_type = kwargs.pop('format_type', 'xml')
        self.codec = kwargs.pop('codec', None)
        super().__init__(**kwargs)



In [10]:
from kraken.configs import TrainingDataConfig
import inspect
print(inspect.getsource(TrainingDataConfig))

class TrainingDataConfig:
    """
    Generic configuration for datasets for all tasks.

    Arg:
        > Universal parameters

        training_data (list of paths):
            A list of training data files.
        evaluation_data (list of paths, optional):
            A list of evaluation data files.
        test_data (list of paths, optional):
            A list of evaluation data files.
        partition (float, defaults to 0.9):
            Automatic partition of training data files if no evaluation data is
            defined.
        num_workers (int, defaults to 1):
            Number of dataloader workers.
        augment (bool, defaults to False):
            Switch to enable augmentation.
        batch_size (int, defaults to 1):
            Number of items to pack into a single sample.
    """
    def __init__(self, **kwargs):
        super().__init__()
        self.training_data = kwargs.pop('training_data', None)
        self.evaluation_data = kwargs.pop('evaluation_da

In [11]:
from kraken.train.vgsl import VGSLRecognitionDataModule
import inspect
# On cherche la methode qui cree train_set (prepare_data ou __init__)
src = inspect.getsource(VGSLRecognitionDataModule)
# Afficher les 100 premieres lignes
print(src[:3000])

/root/htr-medieval-manuscripts-XIVe/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


class VGSLRecognitionDataModule(L.LightningDataModule):
    def __init__(self,
                 data_config: VGSLRecognitionTrainingDataConfig):
        """
        A LightningDataModule encapsulating the training data for a page
        segmentation model.

        Args:
            data_config: Configuration object to set dataset parameters.
        """
        super().__init__()
        self.save_hyperparameters()

        all_files = [getattr(data_config, x) for x in ['training_data', 'evaluation_data', 'test_data']]

        DatasetClass = GroundTruthDataset
        if data_config.format_type in ['xml', 'page', 'alto']:
            if data_config.binary_dataset_split:
                logger.warning('Internal binary dataset splits are enabled but using non-binary dataset files. Will be ignored.')
                data_config.binary_dataset_split = False

            def _parse_xml_set(ds_type, dataset) -> list[dict[str, Segmentation]]:
                if not dataset:
               

In [12]:
import torch
torch.set_float32_matmul_precision('high')

from kraken.configs import VGSLRecognitionTrainingConfig, VGSLRecognitionTrainingDataConfig
from kraken.train.vgsl import VGSLRecognitionModel, VGSLRecognitionDataModule
from kraken.lib.dataset import ArrowIPCRecognitionDataset
from torch.utils.data import Subset
import lightning as L

# 1. Construire les datasets directement avec split_filter
train_ds = ArrowIPCRecognitionDataset(split_filter='train')
train_ds.add('../dataset_nlp/binary_data/train_fixed.arrow')
print(f"Train dataset: {len(train_ds)} lignes")

val_ds = ArrowIPCRecognitionDataset(split_filter='validation')
val_ds.add('../dataset_nlp/binary_data/val_fixed.arrow')
print(f"Val dataset: {len(val_ds)} lignes")

# 2. Construire le data_config minimal (juste pour les hyperparams)
data_config = VGSLRecognitionTrainingDataConfig(
    format_type='binary',
    binary_dataset_split=True,
    training_data=['../dataset_nlp/binary_data/train_fixed.arrow'],
    evaluation_data=['../dataset_nlp/binary_data/val_fixed.arrow'],
    partition=1.0,
    batch_size=8,
    augment=False,
)

train_config = VGSLRecognitionTrainingConfig(
    quit='fixed',
    epochs=1,
    resize='both',
    load='../models/cremma-medieval.mlmodel',
    output='../models/finetune_cremma_v1/model',
    data_config=data_config,
)

# 3. Injecter les datasets dans le data_module APRES instanciation
data_module = VGSLRecognitionDataModule(data_config)
data_module.train_set = Subset(train_ds, range(len(train_ds)))
data_module.val_set = Subset(val_ds, range(len(val_ds)))
print(f"train_set injecte: {len(data_module.train_set)} lignes")
print(f"val_set injecte: {len(data_module.val_set)} lignes")

# 4. Lancer l'entrainement
model = VGSLRecognitionModel(train_config)
trainer = L.Trainer(max_epochs=1, accelerator='gpu', devices=1)
trainer.fit(model, data_module)

Train dataset: 23444 lignes
Val dataset: 5954 lignes


'NoneType' object has no attribute 'valid_norm'
'NoneType' object has no attribute 'valid_norm'


train_set injecte: 23444 lignes
val_set injecte: 5954 lignes


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/root/htr-medieval-manuscripts-XIVe/venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelChec

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃   ┃ Name    ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃                In sizes ┃              Out sizes ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0 │ val_cer │ CharErrorRate  │      0 │ train │     0 │                       ? │                      ? │
│ 1 │ val_wer │ WordErrorRate  │      0 │ train │     0 │                       ? │                      ? │
│ 2 │ net     │ TorchVGSLModel │  4.0 M │ train │ 1.6 B │ [[1, 1, 120, 400], '?'] │ [[1, 140, 1, 50], '?'] │
└───┴─────────┴────────────────┴────────┴───────┴───────┴─────────────────────────┴────────────────────────┘

...

/root/htr-medieval-manuscripts-XIVe/venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/root/htr-medieval-manuscripts-XIVe/venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_con
nector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

Seed set to 42


/root/htr-medieval-manuscripts-XIVe/venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_con
nector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the
value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

Seed set to 42
`Trainer.fit` stopped: `max_epochs=1` reached.


In [13]:
import pandas as pd
import glob
import os

# Lightning sauvegarde les metriques dans lightning_logs/
csv_files = glob.glob('../lightning_logs/*/metrics.csv')
print("Fichiers de logs trouvés:", csv_files)

if csv_files:
    df = pd.read_csv(sorted(csv_files)[-1])
    print(df.tail(10))

# Chercher aussi le checkpoint sauvegarde
checkpoints = glob.glob('../models/finetune_cremma_v1/**/*.ckpt', recursive=True)
print("\nCheckpoints:", checkpoints)
mlmodels = glob.glob('../models/finetune_cremma_v1/**/*.mlmodel', recursive=True)
print("Modeles:", mlmodels)

Fichiers de logs trouvés: []

Checkpoints: []
Modeles: []


In [14]:
%%bash
# Chercher tous les fichiers de logs et checkpoints crees recemment
find /root/htr-medieval-manuscripts-XIVe -name "metrics.csv" -newer /root/htr-medieval-manuscripts-XIVe/experiments/journal.jsonl 2>/dev/null
find /root/htr-medieval-manuscripts-XIVe -name "*.ckpt" -o -name "*.mlmodel" 2>/dev/null | grep -v venv | grep -v "cremma-medieval\|Tridis"
find /root/htr-medieval-manuscripts-XIVe/experiments -name "lightning_logs" -type d 2>/dev/null
ls /root/htr-medieval-manuscripts-XIVe/experiments/lightning_logs/ 2>/dev/null || echo "Pas de lightning_logs dans experiments/"

/root/htr-medieval-manuscripts-XIVe/experiments/lightning_logs/version_3/metrics.csv
/root/htr-medieval-manuscripts-XIVe/experiments/lightning_logs/version_2/metrics.csv
/root/htr-medieval-manuscripts-XIVe/models/mon_modele_finetune/modele_V1/checkpoint_abort.ckpt
/root/htr-medieval-manuscripts-XIVe/models/finetune_cremma_test/model/checkpoint_abort.ckpt
/root/htr-medieval-manuscripts-XIVe/experiments/model/checkpoint_abort.ckpt
/root/htr-medieval-manuscripts-XIVe/experiments/lightning_logs/version_3/checkpoints/epoch=0-step=23444.ckpt
/root/htr-medieval-manuscripts-XIVe/experiments/lightning_logs/version_2/checkpoints/epoch=0-step=23444.ckpt
/root/htr-medieval-manuscripts-XIVe/experiments/lightning_logs
version_0
version_1
version_2
version_3


In [15]:
import pandas as pd

df = pd.read_csv('../experiments/lightning_logs/version_2/metrics.csv')
print(df.to_string())

     epoch   step  train_loss_epoch  train_loss_step  val_accuracy  val_metric  val_word_accuracy
0        0     49               NaN        49.858826           NaN         NaN                NaN
1        0     99               NaN       157.668533           NaN         NaN                NaN
2        0    149               NaN       177.546646           NaN         NaN                NaN
3        0    199               NaN       429.410950           NaN         NaN                NaN
4        0    249               NaN       103.004349           NaN         NaN                NaN
5        0    299               NaN       101.537857           NaN         NaN                NaN
6        0    349               NaN       129.135880           NaN         NaN                NaN
7        0    399               NaN       133.310669           NaN         NaN                NaN
8        0    449               NaN       148.255432           NaN         NaN                NaN
9        0    499   

In [ ]:
import torch
torch.set_float32_matmul_precision('high')

from kraken.configs import VGSLRecognitionTrainingConfig, VGSLRecognitionTrainingDataConfig
from kraken.train.vgsl import VGSLRecognitionModel, VGSLRecognitionDataModule
from kraken.lib.dataset import ArrowIPCRecognitionDataset
from torch.utils.data import Subset
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

train_ds = ArrowIPCRecognitionDataset(split_filter='train')
train_ds.add('../dataset_nlp/binary_data/train_fixed.arrow')

val_ds = ArrowIPCRecognitionDataset(split_filter='validation')
val_ds.add('../dataset_nlp/binary_data/val_fixed.arrow')

data_config = VGSLRecognitionTrainingDataConfig(
    format_type='binary',
    binary_dataset_split=True,
    training_data=['../dataset_nlp/binary_data/train_fixed.arrow'],
    evaluation_data=['../dataset_nlp/binary_data/val_fixed.arrow'],
    partition=1.0,
    batch_size=8,
    augment=False,
)

train_config = VGSLRecognitionTrainingConfig(
    quit='early',
    epochs=-1,
    resize='both',
    load='../models/cremma-medieval.mlmodel',
    output='../models/finetune_cremma_v1/model',
    data_config=data_config,
)

data_module = VGSLRecognitionDataModule(data_config)
data_module.train_set = Subset(train_ds, range(len(train_ds)))
data_module.val_set = Subset(val_ds, range(len(val_ds)))

model = VGSLRecognitionModel(train_config)

checkpoint_cb = ModelCheckpoint(
    dirpath='../models/finetune_cremma_v1/',
    filename='model-{epoch:02d}-{val_metric:.4f}',
    monitor='val_metric',
    mode='max',
    save_top_k=3,
)
early_stop_cb = EarlyStopping(
    monitor='val_metric',
    patience=5,
    mode='max',
    min_delta=0.001,
)

trainer = L.Trainer(
    max_epochs=50,
    accelerator='gpu',
    devices=1,
    callbacks=[checkpoint_cb, early_stop_cb],
)
trainer.fit(model, data_module)
print(f"\nMeilleur modele : {checkpoint_cb.best_model_path}")
print(f"Meilleur val_metric : {checkpoint_cb.best_model_score}")